In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

# 1. Loading and manipulating Brazilian GDP data
df_br_gdp_raw = pd.read_excel('brazil_gdp-data.xlsx')
print(df_br_gdp_raw.T.head(10)) # Finding which columns will be used
df_br_gdp = df_br_gdp_raw.T.iloc[:, [2, 4]] # Taking the right columns
df_br_gdp.columns = ['Period', 'Index']
df_br_gdp['Index'] = pd.to_numeric(df_br_gdp['Index'], errors = 'coerce')

def convert_quartely_ibge(text):
    try:
        parts = str(text).split()
        quarter = int(parts[0][0])
        year = parts[-1]
        month = (quarter * 3) - 2
        return pd.to_datetime(f"{year}-{month:02d}-01")
    except: 
        return pd.NaT

if 'Period' in df_br_gdp.columns:
    df_br_gdp['Quarter_Date'] = df_br_gdp['Period'].apply(convert_quartely_ibge)
    df_br_gdp = df_br_gdp.dropna(subset=['Quarter_Date']).set_index('Quarter_Date')

print(df_br_gdp.tail())

# 2. Converting into Log-Diff and procedure of not detecting outliers
# The reason of not detecting outlier is because the pandemic period, which the volatility is extremely high
df_br_gdp['y'] = np.log(df_br_gdp['Index']).diff()*100
df_br_gdp = df_br_gdp.replace([np.inf, -np.inf], np.nan).dropna(subset = ['y'])
std_dev = df_br_gdp['y'].std()
df_br_gdp['y'] = df_br_gdp['y'].clip(lower = -3*std_dev, upper = 3*std_dev)

# 3. Identifying 2 Regimes by Chain Markov Methodology
np.random.seed(42)

chain_markov_br_gdp = MarkovRegression(df_br_gdp['y'], k_regimes = 2, trend = 'c', switching_variance = False)
chain_markov_br_gdp.initialization = 'approximate-diffuse'
chain_markov_br_gdp_result = chain_markov_br_gdp.fit(em_iter = 0, method = 'powell', search_reps = 100)

# Identificamos qual regime tem a menor média de crescimento (constante)
# Isso garante que idx_rec seja sempre o regime de contração/recessão
idx_rec = chain_markov_br_gdp_result.params.filter(like='const').argmin()

# Mapeamos os nomes com base nesse índice dinâmico
Regime_Name = {idx_rec: "Recession", 1 - idx_rec: "Expansion"}

# Agora a duração será impressa com o rótulo correto
Duration = chain_markov_br_gdp_result.expected_durations
print("\nEXPECTED DURATION OF REGIMES:")
for i, dur in enumerate(Duration):
    print(f"{Regime_Name[i]}: {dur:.2f} quarter(s)")

# 4. Extraction the recession quarters (Prob > .5)
idx_rec = np.argmin(chain_markov_br_gdp_result.params[:2])
rec_probs = chain_markov_br_gdp_result.smoothed_marginal_probabilities[idx_rec]
is_recession = rec_probs > .5

recession_periods = []
start_date = None
for date, value in is_recession.items():
    if value and start_date is None:
        start_date = date
    elif not value and start_date is not None:
        end_date = date
        recession_periods.append((start_date, end_date))
        start_date = None

# 5. Creating chart
plt.figure(figsize = (15, 8))

rec_probs = chain_markov_br_gdp_result.smoothed_marginal_probabilities[0]

# Ploting the GDP variation
plt.plot(df_br_gdp.index, df_br_gdp['Index'], color = 'darkblue', lw = 2.5, label = 'Brazilian GDP (Index)')

params_const = chain_markov_br_gdp_result.params.filter(like='const').sort_values()
r_rec = int(params_const.index[0].split('[')[1][0])
r_exp = int(params_const.index[1].split('[')[1][0])

# Adding shadow
plt.fill_between(df_br_gdp.index, df_br_gdp['Index'].min(), df_br_gdp['Index'].max(),
                 where = (rec_probs > .5), color = 'red', alpha = .2, label = 'Recession')

plt.title('Identification of Economic Regimes in Brazil: Two-State Markov Switching Model', fontsize = 14, fontweight = 'bold')
plt.xlabel("Quarter")
plt.ylabel("Brazilian GDP - Index")
plt.legend(loc = 'best', handlelength = 1.5, frameon = False)
plt.text(0.99, -0.12, 'Source: IBGE - Brazil', transform = plt.gca().transAxes, fontsize = 10, color = 'gray', style = 'italic', horizontalalignment = 'right')
plt.savefig('identification_economic_regimes_two_state.png', dpi = 300, bbox_inches = 'tight')
plt.show()

# 6. Cronology of Regimes
df_br_gdp['Regime_ID'] = chain_markov_br_gdp_result.smoothed_marginal_probabilities.idxmax(axis = 1).values

# Mapping and defining the real regimes
regime_maps = {r_rec: "Recession", r_exp: "Expansion"}
df_br_gdp['Regime_Label'] = df_br_gdp['Regime_ID'].map(regime_maps)

# Creating the Chronology Table
regimes_change = df_br_gdp['Regime_ID'] != df_br_gdp['Regime_ID'].shift()
df_br_gdp['Regime_ID'] = regimes_change.cumsum()

chronology = df_br_gdp.reset_index().groupby('Regime_ID').agg(Regime = ('Regime_Label', 'first'), Beginning = ('Quarter_Date', 'first'), 
                                                             Ending = ('Quarter_Date', 'last'), Quarters = ('Regime_Label', 'count'))

chronology['Beginning'] = pd.PeriodIndex(chronology['Beginning'], freq = 'Q').astype(str)
chronology['Ending'] = pd.PeriodIndex(chronology['Ending'], freq = 'Q').astype(str)

print("\nQUARTERLY CHRONOLOGY - TWO REGIMES")
print(chronology.to_string(index = False))

chronology.to_excel('chronology_2_regimes_br_GDP.xlsx', index=False)